# Vault with identities and creds

`.env` holds only the link to a vault. Secrets are named `<platform>/<key>` and a
plugin says which it needs. Here KeePassXC, in a temporary file with a made-up
password, so nothing else is needed.

In [1]:
from amoorion import Amoorion

# Examples stay off your real setup: see Amoorion.init
scratch = Amoorion.init()

In [2]:
from pykeepass import PyKeePass

from amoorion import Plugins

from amoorion.plugins import load_plugins

from amoorion.vault import VaultLink

from amoorion.vault import kpxc_db_file

filepath = scratch / "identity.kdbx"

link = VaultLink(
    kind="keepassxc",
    path=str(filepath),
    token="made-up",
)

kpxc_db_file.initialize(
    link,
    {"ntfy": load_plugins().get("ntfy").schema()},
)

['/tmp/amoorion-gx_ozg0e/identity.kdbx',
 'group amoorion',
 'entry ntfy',
 'ntfy/topic',
 'ntfy/uri',
 'ntfy/token',
 'ntfy: notes']

That is what `task vault:new` does. Each entry's notes say what its fields are:

In [3]:
entry = PyKeePass(
    filepath,
    password="made-up",
).find_entries(
    title="ntfy",
    first=True,
)

print(entry.notes)

One topic of an ntfy server

topic: The topic; on a public server anyone who knows it can read, so pick a long one
uri (optional): The server; default: https://ntfy.sh
token (optional): Access token, for protected topics


An identity is one env file pointing at one vault. Whether the secrets are there
shows as a flag, and values never:

In [4]:
env_file = scratch / "identity.env"

env_file.write_text(
    f"AMOORION_VAULT=keepassxc\nAMOORION_VAULT_PATH={filepath}\nAMOORION_VAULT_TOKEN=made-up\n"
)

amo = Amoorion(env_file)


def ntfy_ready():

    return next(
        status["ready"]
        for status in amo.accounts()
        if status["platform"] == "ntfy"
    )


print(
    ntfy_ready(),
    Plugins["ntfy"].has_secrets(amo.vault),
)

False False


In [5]:
db = PyKeePass(
    filepath,
    password="made-up",
)

db.find_entries(
    title="ntfy",
    first=True,
).set_custom_property(
    "topic",
    "my-topic",
)

db.save()

amo = Amoorion(env_file)

print(
    ntfy_ready(),
    Plugins["ntfy"].has_secrets(amo.vault),
)

True True


`Amoorion("work.env")` and `Amoorion("private.env")` can live side by side:
`examples/advanced-usecases/scoping-of-identities-handling/`.